# NovaTech — Parte 2: Setup de BigQuery

En este notebook creo el **dataset** y las **8 tablas** del modelo (ver `docs/er_diagram.png` y `docs/normalizacion.md` para el diseño y la justificación 3NF), respetando el orden correcto de claves foráneas (primero las tablas sin dependencias, luego las que dependen de ellas).



In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from google.cloud import bigquery
from google.oauth2 import service_account

# find_dotenv() busca el .env subiendo desde el directorio actual, así que
# funciona tanto si el notebook se ejecuta desde su propia carpeta como si
# se ejecuta desde la raíz del proyecto.
dotenv_path = find_dotenv()
load_dotenv(dotenv_path)
PROJECT_ROOT = Path(dotenv_path).parent if dotenv_path else Path.cwd()

PROJECT_ID = os.getenv("GCP_PROJECT_ID")
DATASET_ID = os.getenv("BQ_DATASET_ID", "novatech")
CREDENTIALS_PATH = os.getenv("GOOGLE_APPLICATION_CREDENTIALS")

assert PROJECT_ID, "Falta GCP_PROJECT_ID en el .env"
assert CREDENTIALS_PATH, "Falta GOOGLE_APPLICATION_CREDENTIALS en el .env"

credentials = service_account.Credentials.from_service_account_file(PROJECT_ROOT / CREDENTIALS_PATH)
client = bigquery.Client(project=PROJECT_ID, credentials=credentials)

print(f"Cliente BigQuery listo -> proyecto={PROJECT_ID}, dataset={DATASET_ID}")

c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Cliente BigQuery listo -> proyecto=proyecto-507914, dataset=novatech


c:\Users\Usuario\AppData\Local\Programs\Python\Python310\lib\site-packages\google\api_core\_python_version_support.py:254: FutureWarning: You are using a Python version (3.10.11) which Google will stop supporting in new releases of google.cloud.bigquery once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.bigquery past that date.
  warnings.warn(message, FutureWarning)


## 1. Crear el dataset

In [3]:
dataset_ref = bigquery.Dataset(f"{PROJECT_ID}.{DATASET_ID}")
dataset_ref.location = "EU"
dataset = client.create_dataset(dataset_ref, exists_ok=True)
print(f"Dataset '{DATASET_ID}' listo en {dataset.location}")

Dataset 'novatech' listo en EU


## 2. Definir los esquemas

El orden de creación importa: primero las tablas "padre" (sin FKs), luego las que dependen de ellas.
BigQuery no obliga las FKs (no es una base de datos transaccional clásica), así que mantengo yo el **orden lógico** y controlo la **integridad referencial** a mano en el código.

In [4]:
SCHEMAS = {
    "categories": [
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("description", "STRING", mode="NULLABLE"),
    ],
    "customers": [
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("first_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("last_name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("email", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("country", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("city", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("acquisition_channel", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("registration_date", "DATE", mode="REQUIRED"),
    ],
    "products": [
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("category_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("name", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("sku", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("unit_price", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("unit_cost", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("stock_quantity", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("is_active", "BOOL", mode="REQUIRED"),
    ],
    "orders": [
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_date", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
    ],
    "order_items": [
        bigquery.SchemaField("order_item_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("quantity", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("unit_price", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("unit_cost", "FLOAT64", mode="REQUIRED"),
    ],
    "payments": [
        bigquery.SchemaField("payment_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("amount", "FLOAT64", mode="REQUIRED"),
        bigquery.SchemaField("payment_date", "TIMESTAMP", mode="REQUIRED"),
        bigquery.SchemaField("method", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
    ],
    "shipments": [
        bigquery.SchemaField("shipment_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("order_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("carrier", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("status", "STRING", mode="REQUIRED"),
        bigquery.SchemaField("shipped_date", "TIMESTAMP", mode="NULLABLE"),
        bigquery.SchemaField("delivered_date", "TIMESTAMP", mode="NULLABLE"),
    ],
    "reviews": [
        bigquery.SchemaField("review_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("product_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("customer_id", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("rating", "INT64", mode="REQUIRED"),
        bigquery.SchemaField("comment", "STRING", mode="NULLABLE"),
        bigquery.SchemaField("review_date", "DATE", mode="REQUIRED"),
    ],
}

# Orden de creación: tablas sin FK primero, luego las que dependen de ellas
CREATION_ORDER = [
    "categories", "customers",              # sin dependencias
    "products",                              # depende de categories
    "orders",                                # depende de customers
    "order_items", "payments", "shipments",  # dependen de orders (y products)
    "reviews",                               # depende de products y customers
]

print(f"{len(SCHEMAS)} tablas definidas, orden de creación: {CREATION_ORDER}")

8 tablas definidas, orden de creación: ['categories', 'customers', 'products', 'orders', 'order_items', 'payments', 'shipments', 'reviews']


## 3. Crear las tablas

In [5]:
for table_name in CREATION_ORDER:
    table_ref = f"{PROJECT_ID}.{DATASET_ID}.{table_name}"
    table = bigquery.Table(table_ref, schema=SCHEMAS[table_name])
    table = client.create_table(table, exists_ok=True)
    print(f"Tabla '{table_name}' lista ({len(SCHEMAS[table_name])} columnas)")

Tabla 'categories' lista (3 columnas)
Tabla 'customers' lista (8 columnas)
Tabla 'products' lista (8 columnas)
Tabla 'orders' lista (4 columnas)
Tabla 'order_items' lista (6 columnas)
Tabla 'payments' lista (6 columnas)
Tabla 'shipments' lista (6 columnas)
Tabla 'reviews' lista (6 columnas)


## 4. Verificación

In [6]:
tables = list(client.list_tables(f"{PROJECT_ID}.{DATASET_ID}"))
print(f"Tablas en el dataset '{DATASET_ID}':")
for t in tables:
    print(" -", t.table_id)

assert {t.table_id for t in tables} == set(SCHEMAS.keys()), "Faltan tablas por crear"
print("\nDataset y tablas verificados correctamente. Listo para 02_generate_data.ipynb")

Tablas en el dataset 'novatech':
 - categories
 - customers
 - order_items
 - orders
 - payments
 - products
 - reviews
 - shipments

Dataset y tablas verificados correctamente. Listo para 02_generate_data.ipynb
